# Indic Subtitle Generator — V1.1 (Colab Edition)

### What this notebook does
| Step | What happens |
|---|---|
| **Cell 1** | Connect Google Drive + check GPU |
| **Cell 2** | Install all dependencies |
| **Cell 3** | Load Whisper + NLLB models (once) |
| **Cell 4** | Upload your video / pick from Drive |
| **Cell 5** | Run the full pipeline (transcribe → translate → embed) |
| **Cell 6** | Download the finished `.mkv` back to your computer |

### Before you start
1. **Enable GPU**: `Runtime → Change runtime type → T4 GPU` (free) or A100 (Colab Pro)
2. Run cells **in order top to bottom** — don't skip any
3. First run downloads ~2 GB of models — subsequent runs are instant
4. Output `.mkv` is saved to **Google Drive** so it survives session resets

---
**Sync fixes in v1.1:**
- Word-level timestamps → subtitle appears **with** the spoken word
- ASS format inside MKV → **instant** track switching in VLC (no 2–10s blank gap)
- Single subtitle line only — no double-line overlap

In [ ]:
# @title ▶ CELL 1 — Connect Google Drive & Check GPU { display-mode: "form" }
# @markdown Run this first. It mounts Google Drive so your output MKV is saved
# @markdown there permanently (survives Colab disconnections).

import os, subprocess, sys

# ── Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

GDRIVE_OUTPUT = '/content/drive/MyDrive/Indic_Subtitles'
os.makedirs(GDRIVE_OUTPUT, exist_ok=True)
print(f'Google Drive mounted')
print(f'   Output folder: {GDRIVE_OUTPUT}')

# ── GPU check ────────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    name   = torch.cuda.get_device_name(0)
    vram   = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'\n GPU: {name}  ({vram:.1f} GB VRAM)')
    DEVICE = 'cuda'
else:
    print('\n  No GPU found!')
    print('   Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    print('   Then: Runtime → Restart session, and run from Cell 1 again.')
    DEVICE = 'cpu'

print(f'\n   Working device: {DEVICE}')
print('    Ready — run Cell 2 next')

In [ ]:
# @title ▶ CELL 2 — Install Dependencies { display-mode: "form" }
# @markdown Takes ~2 minutes on first run. Safe to re-run — skips already-installed packages.

import subprocess, sys, importlib, shutil

PACKAGES = [
    'faster-whisper',
    'ctranslate2',
    'transformers',
    'sentencepiece',
    'tokenizers',
    'tqdm',
    'huggingface_hub',
    'colorama',
]

print('Installing / checking packages...')
for pkg in PACKAGES:
    mod = pkg.replace('-', '_')
    if importlib.util.find_spec(mod) is None:
        print(f'   Installing {pkg}...', end=' ', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        print('done')
    else:
        print(f'  ✔  {pkg}')

# ffmpeg (usually pre-installed on Colab)
if shutil.which('ffmpeg'):
    print(f'  ✔  ffmpeg @ {shutil.which("ffmpeg")}')
else:
    print('  Installing ffmpeg via apt...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], capture_output=True)
    print(f'  ✔  ffmpeg installed')

print('\n All dependencies ready — run Cell 3 next')

In [ ]:
# @title ▶ CELL 3 — Load All Code & Models { display-mode: "form" }
# @markdown Loads Whisper large-v3 (~1.5 GB) and NLLB-200 (~600 MB).
# @markdown First run: ~3-5 min download. Subsequent runs: ~30 seconds.
# @markdown Models are cached in /content/drive/MyDrive/Indic_Subtitles/_models/

import os, re, json, shutil, subprocess, time, warnings
warnings.filterwarnings('ignore')

import torch
from tqdm.notebook import tqdm
from faster_whisper import WhisperModel
import ctranslate2
from ctranslate2.converters import TransformersConverter
from transformers import NllbTokenizerFast

# ── Language maps ─────────────────────────────────────────────────────────
INDIC_LANGUAGES = {
    'Hindi':'hin_Deva','Bengali':'ben_Beng','Telugu':'tel_Telu','Marathi':'mar_Deva',
    'Tamil':'tam_Taml','Urdu':'urd_Arab','Gujarati':'guj_Gujr','Kannada':'kan_Knda',
    'Odia':'ory_Orya','Malayalam':'mal_Mlym','Punjabi':'pan_Guru','Assamese':'asm_Beng',
    'Maithili':'mai_Deva','Sanskrit':'san_Deva','Nepali':'npi_Deva','Sindhi':'snd_Arab',
    'Kashmiri':'kas_Arab','Konkani':'kok_Deva','Dogri':'doi_Deva','Santali':'sat_Olck',
    'Meitei':'mni_Mtei','Bhojpuri':'bho_Deva','English':'eng_Latn',
}
WHISPER_TO_NLLB = {
    'hi':'hin_Deva','en':'eng_Latn','bn':'ben_Beng','te':'tel_Telu','mr':'mar_Deva',
    'ta':'tam_Taml','ur':'urd_Arab','gu':'guj_Gujr','kn':'kan_Knda','or':'ory_Orya',
    'ml':'mal_Mlym','pa':'pan_Guru','as':'asm_Beng','ne':'npi_Deva','sa':'san_Deva',
    'sd':'snd_Arab','ks':'kas_Arab','si':'sin_Sinh',
}
NLLB_TO_NAME = {v: k for k, v in INDIC_LANGUAGES.items()}
LANG_ISO = {
    'Hindi':'hin','Bengali':'ben','Telugu':'tel','Marathi':'mar','Tamil':'tam',
    'Urdu':'urd','Gujarati':'guj','Kannada':'kan','Odia':'ori','Malayalam':'mal',
    'Punjabi':'pan','Assamese':'asm','Maithili':'mai','Sanskrit':'san','Nepali':'nep',
    'Sindhi':'snd','Kashmiri':'kas','Konkani':'kok','Dogri':'doi','Santali':'sat',
    'Meitei':'mni','Bhojpuri':'bho','English':'eng',
}

# ── Tuning constants ──────────────────────────────────────────────────────
SYNC_LEAD_MS       = 350
MAX_SUB_SECS       = 4.0
MIN_GAP_MS         = 50
MAX_CHARS_PER_LINE = 60

# ── ASS helpers ───────────────────────────────────────────────────────────
def _ass_time(sec):
    sec = max(0.0, sec)
    cs  = int(round(sec * 100)) % 100
    s   = int(sec) % 60
    m   = (int(sec) // 60) % 60
    h   = int(sec) // 3600
    return f'{h}:{m:02d}:{s:02d}.{cs:02d}'

_ASS_HEADER = '''[Script Info]
ScriptType: v4.00+
PlayResX: 1280
PlayResY: 720
ScaledBorderAndShadow: yes
WrapStyle: 0
Collisions: Normal

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,42,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,0,0,0,0,100,100,0,0,1,2,1,2,20,20,30,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
'''

def write_ass(segments, path):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(_ASS_HEADER)
        for seg in segments:
            text = seg['text'].strip()
            if not text: continue
            text = re.sub(r'\s*\n\s*', ' ', text).strip()
            if len(text) > MAX_CHARS_PER_LINE:
                text = text[:MAX_CHARS_PER_LINE].rsplit(' ', 1)[0] + '…'
            f.write(f"Dialogue: 0,{_ass_time(seg['start'])},{_ass_time(seg['end'])},Default,,0,0,0,,{text}\n")

def read_ass(path):
    segs, in_ev, fmt = [], False, []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.strip() == '[Events]': in_ev = True; continue
            if in_ev:
                if line.startswith('Format:'):
                    fmt = [x.strip() for x in line[7:].split(',')]
                elif line.startswith('Dialogue:'):
                    parts = line[9:].split(',', len(fmt)-1)
                    if len(parts) < len(fmt): continue
                    d = dict(zip(fmt, parts))
                    try:
                        def _p(t):
                            t=t.strip(); h,m,sc=t.split(':'); s,cs=sc.split('.')
                            return int(h)*3600+int(m)*60+int(s)+int(cs)/100
                        segs.append({'start':_p(d['Start']),'end':_p(d['End']),'text':d.get('Text','').strip()})
                    except: pass
    return segs

def build_word_segments(raw_segments, lead_ms=SYNC_LEAD_MS):
    lead_s = lead_ms / 1000.0
    gap_s  = MIN_GAP_MS / 1000.0
    pause_thr = 0.30
    words = []
    for seg in raw_segments:
        if seg.get('words'):
            for w in seg['words']:
                if w.word.strip():
                    words.append({'word':w.word.strip(),'start':w.start,'end':w.end})
        else:
            words.append({'word':seg['text'].strip(),'start':seg['start'],'end':seg['end']})
    if not words: return []
    groups, cur = [], [words[0]]
    for w in words[1:]:
        prev  = cur[-1]
        pause = w['start'] - prev['end']
        dur   = w['end'] - cur[0]['start']
        chars = sum(len(x['word']) for x in cur) + len(cur) + len(w['word']) + 1
        if dur > MAX_SUB_SECS or chars > MAX_CHARS_PER_LINE or pause > pause_thr:
            groups.append(cur); cur = [w]
        else:
            cur.append(w)
    if cur: groups.append(cur)
    segs = []
    for grp in groups:
        segs.append({
            'start': round(max(0.0, grp[0]['start'] - lead_s), 3),
            'end':   round(grp[-1]['end'], 3),
            'text':  ' '.join(x['word'] for x in grp).strip(),
        })
    for i in range(len(segs)-1):
        mx = segs[i+1]['start'] - gap_s
        if segs[i]['end'] > mx:
            segs[i]['end'] = round(max(segs[i]['start']+0.1, mx), 3)
    return segs

def translate_batch(texts, src_lang, tgt_lang, batch_size=32, beam_size=2):
    if not texts: return []
    _nllb_tokenizer.src_lang = src_lang
    results = []
    for i in range(0, len(texts), batch_size):
        batch     = texts[i:i+batch_size]
        valid_idx = [j for j,t in enumerate(batch) if t.strip()]
        valid_txt = [batch[j] for j in valid_idx]
        out       = list(batch)
        if not valid_txt: results.extend(out); continue
        try:
            enc = _nllb_tokenizer(valid_txt, return_tensors=None,
                                  padding=False, truncation=True, max_length=256)
            tb  = [_nllb_tokenizer.convert_ids_to_tokens(ids) for ids in enc['input_ids']]
            tr  = _nllb_translator.translate_batch(
                tb, target_prefix=[[tgt_lang]]*len(tb),
                beam_size=beam_size, max_decoding_length=256,
                repetition_penalty=1.2, no_repeat_ngram_size=4)
            for j,res in zip(valid_idx, tr):
                toks = res.hypotheses[0]
                if toks and toks[0]==tgt_lang: toks=toks[1:]
                dec = _nllb_tokenizer.decode(
                    _nllb_tokenizer.convert_tokens_to_ids(toks),
                    skip_special_tokens=True, clean_up_tokenization_spaces=True).strip()
                out[j] = dec if dec else batch[j]
        except Exception as e:
            print(f'    batch error: {e}')
        results.extend(out)
    return results

def get_duration(video_path):
    r = subprocess.run(['ffprobe','-v','quiet','-print_format','json',
                        '-show_format', video_path], capture_output=True, text=True)
    if r.returncode==0:
        return float(json.loads(r.stdout).get('format',{}).get('duration',0))
    return 0.0

# ── Model paths (cached on Drive so downloads survive session resets) ─────
NLLB_CACHE_DIR = '/content/drive/MyDrive/Indic_Subtitles/_models/nllb_ct2_int8'
NLLB_HF_NAME   = 'facebook/nllb-200-distilled-600M'
os.makedirs(NLLB_CACHE_DIR, exist_ok=True)

# ── Load Whisper ──────────────────────────────────────────────────────────
print('\n── Loading Whisper large-v3 ──────────────────────────────')
ct = 'float16' if DEVICE=='cuda' else 'int8'
t0 = time.time()
_whisper_model = WhisperModel('large-v3', device=DEVICE, compute_type=ct)
print(f' Whisper loaded in {time.time()-t0:.1f}s')

# ── Load / convert NLLB ───────────────────────────────────────────────────
print('\n── Loading NLLB-200-distilled-600M ──────────────────────────')
if not os.path.exists(os.path.join(NLLB_CACHE_DIR, 'model.bin')):
    print('  First run: downloading + converting NLLB (~3-5 min)...')
    t0   = time.time()
    conv = TransformersConverter(NLLB_HF_NAME, low_cpu_mem_usage=True)
    conv.convert(output_dir=NLLB_CACHE_DIR, quantization='int8', force=True)
    print(f'   Converted in {time.time()-t0:.0f}s — cached to Drive')
else:
    print('   NLLB cache found on Drive — loading directly')

ct2 = 'int8_float16' if DEVICE=='cuda' else 'int8'
t0  = time.time()
_nllb_translator = ctranslate2.Translator(NLLB_CACHE_DIR, device=DEVICE,
                                           compute_type=ct2,
                                           inter_threads=1, intra_threads=4)
_nllb_tokenizer  = NllbTokenizerFast.from_pretrained(NLLB_HF_NAME)
print(f' NLLB loaded in {time.time()-t0:.1f}s')

if DEVICE == 'cuda':
    used = torch.cuda.memory_allocated()/1024**3
    print(f'   VRAM in use: {used:.2f} GB')

print('\n All models ready — run Cell 4 next')

In [ ]:
# @title ▶ CELL 4 — Upload Your Video { display-mode: "form" }
# @markdown Choose how to provide the video file:

UPLOAD_METHOD = "local_path"  # @param ["upload", "local_path", "gdrive"]

# @markdown ---
# @markdown **Option 1: Upload directly (file picker)**
# @markdown   Slower, but works for files <2 GB.

# @markdown **Option 2: Existing Colab path** (file already on this VM)
LOCAL_VIDEO_PATH = "/content/Hindi_1.mp4"  # @param {type:"string"}

# @markdown **Option 3: Google Drive** (mount Drive first in Cell 1)
GDRIVE_VIDEO_PATH = ""  # @param {type:"string"}

import os
from google.colab import files

VIDEO_PATH = None

# ---------- Option 1: Browser upload ----------
if UPLOAD_METHOD == "upload":
    print(' A file picker will appear — select your video file.')
    print('   Supported: .mp4  .mkv  .avi  .mov  .webm  .m4v\n')
    uploaded = files.upload()
    if not uploaded:
        print(' No file selected.')
    else:
        fname = list(uploaded.keys())[0]
        VIDEO_PATH = f'/content/{fname}'

# ---------- Option 2: Existing local path ----------
elif UPLOAD_METHOD == "local_path":
    VIDEO_PATH = LOCAL_VIDEO_PATH
    if not os.path.exists(VIDEO_PATH):
        print(f' File not found: {VIDEO_PATH}')
        print('   Make sure the path is correct and the file already exists in Colab.')
        VIDEO_PATH = None

# ---------- Option 3: Google Drive ----------
elif UPLOAD_METHOD == "gdrive":
    VIDEO_PATH = GDRIVE_VIDEO_PATH
    if not os.path.exists(VIDEO_PATH):
        print(f' File not found: {VIDEO_PATH}')
        print('   Check the path and make sure Drive is mounted (Cell 1)')
        VIDEO_PATH = None
else:
    print(' Invalid upload method selected.')

# ---------- Final check + video info ----------
if VIDEO_PATH and os.path.exists(VIDEO_PATH):
    size_mb = os.path.getsize(VIDEO_PATH) / 1024**2
    print(f'\n Video ready: {VIDEO_PATH}')
    print(f'   Size: {size_mb:.1f} MB')
    # Show duration if the helper function exists
    try:
        dur = get_duration(VIDEO_PATH)
        if dur > 0:
            print(f'   Duration: {int(dur)//60}m {int(dur)%60:02d}s')
    except NameError:
        pass  # get_duration not defined yet – ignore
    print('\n Video ready — run Cell 5 to generate subtitles')
else:
    print('\n No valid video file loaded. Please check your selection and try again.')

In [ ]:
# @title ▶ CELL 5 — Run Full Pipeline { display-mode: "form" }
# @markdown Transcribes audio → translates to all 22 Indian languages + English
# @markdown → embeds all tracks as ASS subtitles → saves `.mkv` to Google Drive.
# @markdown
# @markdown ⏱ Typical times on T4 GPU:
# @markdown - 10-min video: ~12 min total
# @markdown - 30-min video: ~35 min total
# @markdown - 60-min video: ~70 min total

import os, re, json, subprocess, time
from tqdm.notebook import tqdm

if 'VIDEO_PATH' not in dir() or not os.path.exists(VIDEO_PATH):
    print(' No video loaded — run Cell 4 first!')
else:
    t_pipeline = time.time()

    VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
    # Output goes BOTH locally (fast I/O) and copied to Drive at the end
    LOCAL_OUT  = f'/content/{VIDEO_NAME}_subtitles'
    DRIVE_OUT  = f'/content/drive/MyDrive/Indic_Subtitles/{VIDEO_NAME}_subtitles'
    os.makedirs(LOCAL_OUT, exist_ok=True)
    os.makedirs(DRIVE_OUT, exist_ok=True)

    VIDEO_DUR  = get_duration(VIDEO_PATH)
    print(f'{'='*60}')
    print(f'    {VIDEO_NAME}')
    print(f'{'='*60}')
    print(f'   Duration : {int(VIDEO_DUR)//60}m {int(VIDEO_DUR)%60:02d}s')
    print(f'   Local out: {LOCAL_OUT}')
    print(f'   Drive out: {DRIVE_OUT}')

    # ── Step 1: Extract audio ─────────────────────────────────────────────
    print('\n── Step 1/5: Extracting audio ───────────────────────────')
    AUDIO_PATH = f'{LOCAL_OUT}/{VIDEO_NAME}_audio.wav'
    r = subprocess.run(
        ['ffmpeg','-y','-i',VIDEO_PATH,'-vn','-acodec','pcm_s16le',
         '-ar','16000','-ac','1', AUDIO_PATH],
        capture_output=True, text=True)
    if r.returncode!=0:
        print(' Audio extraction failed:', r.stderr[-500:])
    else:
        print(f' Audio: {os.path.getsize(AUDIO_PATH)/1024**2:.2f} MB')

        # ── Step 2: Transcribe ────────────────────────────────────────────
        print('\n── Step 2/5: Transcribing (word-level timestamps) ───────')
        t1 = time.time()
        segs_gen, meta = _whisper_model.transcribe(
            AUDIO_PATH, beam_size=5, language=None,
            word_timestamps=True,
            condition_on_previous_text=False,
            compression_ratio_threshold=2.4,
            log_prob_threshold=-1.0,
            no_speech_threshold=0.6,
            vad_filter=True,
            vad_parameters={'min_silence_duration_ms':800,'speech_pad_ms':400,'threshold':0.35},
        )

        detected_lang = meta.language
        src_nllb      = WHISPER_TO_NLLB.get(detected_lang, 'hin_Deva')
        src_name      = NLLB_TO_NAME.get(src_nllb, f'Unknown ({detected_lang})')
        print(f'   Detected: {src_name} ("{detected_lang}")  |  {src_nllb}')
        print(f'   Audio duration: {meta.duration:.1f}s')

        raw = []
        with tqdm(total=int(meta.duration), unit='s',
                  desc='  Transcribing', ncols=70) as pbar:
            last = 0.0
            for seg in segs_gen:
                raw.append({'start':seg.start,'end':seg.end,'text':seg.text,'words':seg.words})
                pbar.update(max(0, int(seg.end-last))); last=seg.end

        cov = raw[-1]['end']/meta.duration*100 if raw and meta.duration>0 else 0
        print(f' Transcription done in {time.time()-t1:.1f}s — '
              f'{len(raw)} segments, {cov:.0f}% coverage')

        # ── Step 3: Build word-level segments ─────────────────────────────
        print('\n── Step 3/5: Building sync-corrected subtitle segments ───')
        synced = build_word_segments(raw, lead_ms=SYNC_LEAD_MS)
        print(f'{len(synced)} subtitle lines  |  '
              f'sync lead: {SYNC_LEAD_MS}ms  |  max dur: {MAX_SUB_SECS}s')

        orig_slug    = src_name.split()[0].lower()
        ORIGINAL_ASS = f'{LOCAL_OUT}/{VIDEO_NAME}_{orig_slug}_original.ass'
        write_ass(synced, ORIGINAL_ASS)
        print(f'   Saved original ASS: {orig_slug}_original.ass ({src_name})')

        # ── Step 4: Translate ──────────────────────────────────────────────
        print(f'\n── Step 4/5: Translating to all 22 languages ────────────')
        TARGET_LANGS = {n:c for n,c in INDIC_LANGUAGES.items() if c!=src_nllb}
        orig_segs    = read_ass(ORIGINAL_ASS)
        source_texts = [s['text'] for s in orig_segs]

        GENERATED_ASS = []
        FAILED_LANGS  = []
        t_tr = time.time()

        for i,(lang_name,tgt_code) in enumerate(tqdm(
            list(TARGET_LANGS.items()), desc='  Translating', ncols=70), 1):
            print(f'  [{i:2d}/{len(TARGET_LANGS)}] {lang_name:15s} ({tgt_code}) ',
                  end='', flush=True)
            t0 = time.time()
            try:
                translated = translate_batch(source_texts, src_nllb, tgt_code)
                trans_segs = [
                    {'start':s['start'],'end':s['end'],
                     'text': t if t.strip() else s['text']}
                    for s,t in zip(orig_segs,translated)
                ]
                safe  = lang_name.lower().replace(' ','_')
                apath = f'{LOCAL_OUT}/{VIDEO_NAME}_{safe}.ass'
                write_ass(trans_segs, apath)
                GENERATED_ASS.append((lang_name, apath))
                print(f' {time.time()-t0:.1f}s')
            except Exception as e:
                print(f' {e}')
                FAILED_LANGS.append((lang_name, str(e)))

        print(f'\n Translation done in {time.time()-t_tr:.1f}s')
        if FAILED_LANGS:
            print(f'  Failed: {[l for l,_ in FAILED_LANGS]}')

        # ── Step 5: Mux into MKV ───────────────────────────────────────────
        print(f'\n── Step 5/5: Embedding {1+len(GENERATED_ASS)} ASS tracks into MKV ────')
        ALL_TRACKS = []
        if os.path.exists(ORIGINAL_ASS):
            ALL_TRACKS.append((f'Original-{src_name.split()[0]}',
                               LANG_ISO.get(src_name.split()[0],'und'), ORIGINAL_ASS))
        for lang_name,apath in GENERATED_ASS:
            ALL_TRACKS.append((lang_name, LANG_ISO.get(lang_name,'und'), apath))

        OUTPUT_MKV  = f'{LOCAL_OUT}/{VIDEO_NAME}_all_subtitles.mkv'
        cmd  = ['ffmpeg','-y','-i',VIDEO_PATH]
        for _,_,ass in ALL_TRACKS: cmd += ['-i', ass]
        cmd += ['-map','0:v','-map','0:a?']
        for k in range(len(ALL_TRACKS)): cmd += ['-map', f'{k+1}:0']
        cmd += ['-c:v','copy','-c:a','copy','-c:s','ass']
        for k,(name,iso,_) in enumerate(ALL_TRACKS):
            cmd += [f'-metadata:s:s:{k}', f'title={name}']
            cmd += [f'-metadata:s:s:{k}', f'language={iso}']
        cmd += ['-metadata', f'title={VIDEO_NAME} — Multi-language Subtitles']
        cmd += ['-disposition:s:0','default', OUTPUT_MKV]

        t0 = time.time()
        r  = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(' ffmpeg mux failed:', r.stderr[-800:])
        else:
            print(f' MKV muxed in {time.time()-t0:.1f}s')

            # ── Copy everything to Google Drive ───────────────────────────
            print(f'\n── Copying output to Google Drive ───────────────────────')
            import shutil
            # Copy MKV
            drive_mkv = f'{DRIVE_OUT}/{VIDEO_NAME}_all_subtitles.mkv'
            shutil.copy2(OUTPUT_MKV, drive_mkv)
            # Copy all ASS files
            for fname in os.listdir(LOCAL_OUT):
                if fname.endswith('.ass'):
                    shutil.copy2(f'{LOCAL_OUT}/{fname}', f'{DRIVE_OUT}/{fname}')

            total_t = time.time() - t_pipeline
            out_mb  = os.path.getsize(OUTPUT_MKV)/1024**2

            print(f'\n{'='*60}')
            print(f'    PIPELINE COMPLETE!  ({int(total_t)//60}m {int(total_t)%60:02d}s)')
            print(f'{'='*60}')
            print(f'  Output MKV   : {out_mb:.1f} MB')
            print(f'  Tracks       : {len(ALL_TRACKS)} subtitle languages (ASS)')
            print(f'  Saved to     : {DRIVE_OUT}/')
            print(f'  Local copy   : {OUTPUT_MKV}')
            print()
            print('   VLC: Subtitles menu → Sub Track → pick language')
            print('   MPV: "#" key to cycle tracks')
            print()
            print('   Run Cell 6 to download the MKV directly to your computer')

            # Make path available for Cell 6
            FINAL_MKV  = OUTPUT_MKV
            FINAL_DIR  = LOCAL_OUT

In [ ]:
# @title ▶ CELL 6 — Download Output to Your Computer { display-mode: "form" }
# @markdown Downloads the finished MKV file directly to your browser.
# @markdown If the file is very large (>2 GB), use the Google Drive option instead.

DOWNLOAD_MKV   = True   # @param {type:"boolean"}
DOWNLOAD_ASSES = False  # @param {type:"boolean"}
# @markdown Set `DOWNLOAD_ASSES = True` to also download all individual .ass subtitle files.

from google.colab import files
import os

if 'FINAL_MKV' not in dir():
    print(' No output found — run Cell 5 first!')
else:
    if DOWNLOAD_MKV:
        if os.path.exists(FINAL_MKV):
            sz = os.path.getsize(FINAL_MKV)/1024**2
            print(f'⬇️  Downloading {os.path.basename(FINAL_MKV)}  ({sz:.1f} MB)...')
            print('   (A browser download dialog will appear)')
            files.download(FINAL_MKV)
        else:
            print(f' MKV not found: {FINAL_MKV}')
            print('   It IS saved on Google Drive — check MyDrive/Indic_Subtitles/')

    if DOWNLOAD_ASSES:
        ass_files = sorted(f for f in os.listdir(FINAL_DIR) if f.endswith('.ass'))
        print(f'⬇  Downloading {len(ass_files)} .ass subtitle files...')
        for fname in ass_files:
            print(f'   {fname}')
            files.download(f'{FINAL_DIR}/{fname}')

    print('\n Download started!')
    print('   Files are ALSO saved permanently at:')
    print(f'   Google Drive → MyDrive/Indic_Subtitles/{VIDEO_NAME}_subtitles/')